In [18]:
import importlib

import impish_stack
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u
from astropy.visualization import quantity_support

from adetsim import detection

importlib.reload(detection)

%matplotlib qt
plt.style.use("nice.mplstyle")

In [33]:
de = 0.1
model_bins = np.arange(1, 1000 + de, de) << u.keV
det_stack = impish_stack.generate(scintillator="yap")
ct_conversion = det_stack.compute_absorption(energy_edges=model_bins)

In [34]:
fig, ax = plt.subplots()
ax.stairs(ct_conversion.to_value(u.percent), model_bins.to_value(u.keV))
ax.set(xscale="log", xlabel="energy keV", yscale="linear", ylabel="conversion probability (%)")
plt.show()

In [52]:
de = 0.2
bins = np.arange(10, 300 + de, de)
size = bins.size - 1

# Fake DRM with some off-diagonal terms
example = np.eye(size)
np.fill_diagonal(example[:, 250:], 0.5)
np.fill_diagonal(example[:, 650:], 0.3)

# The probabilities must still sum to 1: we have escape peaks now
example = example / example.sum(axis=0)

fig, ax = plt.subplots()
pcm = ax.pcolormesh(bins, bins, example)
fig.colorbar(pcm, label='probability')
plt.show()

In [53]:
lyso_energies = [60, 31, 122] << u.keV
lyso_fwhms = [30, 50, 21] << u.percent

yap_energies = [31, 60, 122] << u.keV
yap_fwhms = [27.5, 18, 13] << u.percent

energies = yap_energies
fwhms = yap_fwhms

fwhm_error = fwhms * 0.05

ereln = detection.SqrtEnergyResolution(
    reference_fwhms=fwhms, fwhm_errors=fwhm_error, reference_energies=energies
)

In [54]:
fig, ax = plt.subplots()

with quantity_support():
    ax.errorbar(
        energies,
        fwhms,
        yerr=fwhm_error,
        color="red",
        label="FWHMs from data",
        marker=".",
        ms=12,
        capsize=6,
        ls="None",
        zorder=-1,
    )

energy_range = bins
ax.plot(
    energy_range,
    100 * ereln.resolution_function(energy_range),
    label=r"$1 / \sqrt{E}$ fit",
    color="black",
)

ax.legend()
ax.set(
    ylabel="FWHM energy resolution (%)",
    xlabel="Energy (keV)",
    title="Fitting YAP resolutions",
)

plt.show()

In [55]:
resolution_matrix = ereln.generate_resolution_matrix(bins << u.keV)

In [56]:
fig, ax = plt.subplots()
ax.stairs(resolution_matrix.sum(axis=0), bins)
plt.show()

In [57]:
import matplotlib.colors as mcol

# norm = mcol.SymLogNorm(1e-3, vmin=0, vmax=1)
# norm = mcol.LogNorm()
norm = None

fig, ax = plt.subplots()
pcm = ax.pcolormesh(energy_range, energy_range, (drm := resolution_matrix @ example), norm=norm)
# pcm = ax.pcolormesh(energy_range, energy_range, resolution_matrix.T / resolution_matrix, norm=norm)
ax.set(
    xlabel="input energy bins",
    ylabel="output enregy bins",
    title="drm with resolution multiplied in",
)
_ = fig.colorbar(pcm, label="probability")
plt.show()

In [58]:
fig, ax = plt.subplots()
ax.stairs(drm.sum(axis=0), energy_range, label="Resolution applied (counts ?)")
ax.stairs(example.sum(axis=0), energy_range, label="No resolution (photons)")
ax.set(xlabel="Energy keV", ylabel="Deposition probability")
ax.legend()
plt.show()